In [ ]:
# ─────────────────────────────────────────
# 셀 1: 라이브러리 & 모듈 import
# ─────────────────────────────────────────
import sys
import os

sys.path.append('../../models/role04_self')

from self_detector import detect_self_language

# 확인용
print(os.listdir('../../models/role04_self'))

In [ ]:
# ─────────────────────────────────────────
# 셀 2: 더미 문장 + 정답 레이블 정의
# gold: self_hits에 탐지되어야 할 자기중심 표현 목록
# ─────────────────────────────────────────
DUMMY_DATA = [
    (
        "저는 팀 프로젝트에서 리더를 맡았습니다.",
        ["저는"]
    ),
    (
        "3개월간 A/B 테스트를 12회 진행해 전환율을 23% 개선했습니다.",
        []
    ),
    (
        "이 경험을 통해 책임감과 도전정신을 키웠습니다.",
        ["책임감", "도전정신"]
    ),
    (
        "팀의 매출을 30% 향상시켰습니다.",
        []
    ),
    (
        "저는 이 경험을 통해 많은 것을 느꼈습니다.",
        ["저는", "느꼈습니다"]
    ),
    (
        "Python과 SQL을 활용해 고객 이탈률을 15%p 낮췄습니다.",
        []
    ),
    (
        "저는 항상 최선을 다했으며 성실함으로 팀에 기여했습니다.",
        ["저는", "성실함"]
    ),
    (
        "매주 코드 리뷰를 주도해 팀 버그 발생률을 40% 줄였습니다.",
        []
    ),
    (
        "저는 이번 프로젝트를 통해 성장했습니다.",
        ["저는", "성장했습니다"]
    ),
    (
        "도전정신과 열정으로 어려운 문제도 해결했습니다.",
        ["도전정신", "열정"]
    ),
]

print(f"더미 문장 총 {len(DUMMY_DATA)}개 로드 완료")

In [ ]:
# ─────────────────────────────────────────
# 셀 3: 문장별 탐지 결과 출력
# ─────────────────────────────────────────
for i, (sent, gold) in enumerate(DUMMY_DATA, 1):
    result = detect_self_language(sent)
    detected_self = [h["term"] for h in result["self_hits"]]
    detected_contrib = [h["term"] for h in result["contribution_hits"]]

    print(f"\n{'='*60}")
    print(f"[문장 {i}] {sent}")
    print(f"  정답 레이블      : {gold}")
    print(f"  탐지된 자기중심  : {detected_self}")
    print(f"  탐지된 기여중심  : {detected_contrib}")
    print(f"  수치 표현        : {result['number_hits']}")
    print(f"  자기중심 점수    : {result['self_score']}점")
    print(f"  기여중심 점수    : {result['contribution_score']}점")
    print(f"  등급             : {result['grade']} — {result['summary']}")

In [ ]:
# ─────────────────────────────────────────
# 셀 4: 재현율·정밀도·F1 계산
# ─────────────────────────────────────────
total_tp, total_fn, total_fp = 0, 0, 0

for sent, gold in DUMMY_DATA:
    result = detect_self_language(sent)
    detected = [h["term"] for h in result["self_hits"]]

    tp = [g for g in gold    if any(g in d or d in g for d in detected)]
    fn = [g for g in gold    if g not in detected]
    fp = [d for d in detected if not any(d in g or g in d for g in gold)]

    total_tp += len(tp)
    total_fn += len(fn)
    total_fp += len(fp)

recall    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("="*40)
print(f"  TP (맞게 탐지)     : {total_tp}개")
print(f"  FN (놓친 표현)     : {total_fn}개  ← 사전에 추가 필요")
print(f"  FP (오탐)          : {total_fp}개  ← 사전에서 제거 필요")
print("─"*40)
print(f"  재현율  (Recall)   : {recall:.1%}")
print(f"  정밀도  (Precision): {precision:.1%}")
print(f"  F1 Score           : {f1:.1%}")
print("="*40)

In [ ]:
# ─────────────────────────────────────────
# 셀 5: FN 분석 — 놓친 표현 목록
# ─────────────────────────────────────────
print("【 놓친 표현(FN) 목록 — 사전 보강 대상 】\n")

for i, (sent, gold) in enumerate(DUMMY_DATA, 1):
    result = detect_self_language(sent)
    detected = [h["term"] for h in result["self_hits"]]
    fn = [g for g in gold if g not in detected]

    if fn:
        print(f"  문장 {i}: {fn}  →  사전에 추가하세요")

print("\n【 오탐(FP) 목록 — 사전에서 제거 대상 】\n")

for i, (sent, gold) in enumerate(DUMMY_DATA, 1):
    result = detect_self_language(sent)
    detected = [h["term"] for h in result["self_hits"]]
    fp = [d for d in detected if not any(d in g or g in d for g in gold)]

    if fp:
        print(f"  문장 {i}: {fp}  →  사전에서 제거하세요")

In [ ]:
# ─────────────────────────────────────────
# 셀 6: 피드백 출력 확인
# ─────────────────────────────────────────
print("【 피드백 생성 테스트 】\n")

test_sent = "저는 항상 열심히 노력했으며 이 경험을 통해 많은 것을 느꼈고 성장했습니다."
result = detect_self_language(test_sent)

print(f"입력: {test_sent}\n")
print(f"자기중심 점수 : {result['self_score']}점")
print(f"기여중심 점수 : {result['contribution_score']}점")
print(f"등급          : {result['grade']}등급 — {result['summary']}\n")

if result['feedback_items']:
    print("개선 제안:")
    for fb in result['feedback_items']:
        print(f"  [{fb['category']}] '{fb['original']}' {fb['suggestion']}")
else:
    print("피드백 없음 (자기중심 표현 미탐지)")